In [21]:
import sys
from mlflow.tracking import MlflowClient

import numpy as np
import pandas as pd


from tqdm import tqdm

sys.path.append("../.")


pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 400)
pd.set_option("display.float_format", lambda x: "%.4f" % x)


%reload_ext autoreload
%autoreload 2

In [ ]:
# Initialize MLflow Client
client = MlflowClient(tracking_uri="http://potato.felk.cvut.cz:2222")

experiment_names = [
    "pelesjak_getml_xgboost",
    "pelesjak_lightgbm_aggregate_hyperparams",
    "pelesjak_lightgbm_hyperparams",
    "pelesjak_linear_dbformer_hyperparams",
    "pelesjak_linear_sage_hyperparams",
    "pelesjak_linear_tabular_aggregate_hyperparams",
    "pelesjak_linear_tabular_hyperparams",
    "pelesjak_resnet_sage_hyperparams",
    "pelesjak_resnet_tabular_aggregate_hyperparams",
    "pelesjak_resnet_tabular_hyperparams",
]

for experiment_name in experiment_names:
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found.")
    experiment_id = experiment.experiment_id

    # Get all runs for the experiment
    runs = client.search_runs(experiment_ids=[experiment_id], max_results=10000)

    runs_dict = []
    for r in tqdm(runs.to_list()):
        r_info = r.info.__dict__
        r_data = {k: v for d in r.data.to_dictionary().values() for k, v in d.items()}

        runs_dict.append({**r_info, **r_data})

    df = pd.DataFrame(runs_dict)

    df.to_csv(f"./data/{experiment_name}.csv")

In [22]:
dataset_tasks_index = [
    ("ctu-ergastf1", "ergastf1-original"),
    ("ctu-expenditures", "expenditures-original"),
    ("ctu-geneea", "geneea-original"),
    ("ctu-geneea", "geneea-temporal"),
    ("ctu-hepatitis", "hepatitis-original"),
    ("ctu-imdb", "imdb-original"),
    ("ctu-mondial", "mondial-original"),
    ("ctu-movielens", "movielens-original"),
    ("ctu-musklarge", "musklarge-original"),
    ("ctu-musksmall", "musksmall-original"),
    ("ctu-mutagenesis", "mutagenesis-original"),
    ("ctu-ncaa", "ncaa-original"),
    ("ctu-studentloan", "studentloan-original"),
    ("rel-amazon", "item-churn"),
    ("rel-amazon", "user-churn"),
    ("rel-avito", "user-clicks"),
    ("rel-avito", "user-visits"),
    ("rel-f1", "driver-dnf"),
    ("rel-f1", "driver-top3"),
    ("rel-stack", "user-badge"),
    ("rel-stack", "user-engagement"),
    ("rel-trial", "study-outcome"),
    ("ctu-accidents", "accidents-original"),
    ("ctu-accidents", "accidents-temporal"),
    ("ctu-craftbeer", "craftbeer-original"),
    ("ctu-dallas", "dallas-original"),
    ("ctu-dallas", "dallas-temporal"),
    ("ctu-diabetes", "diabetes-original"),
    ("ctu-financial", "financial-original"),
    ("ctu-financial", "financial-temporal"),
    ("ctu-genes", "genes-original"),
    ("ctu-hockey", "hockey-original"),
    ("ctu-legalacts", "legalacts-original"),
    ("ctu-legalacts", "legalacts-temporal"),
    ("ctu-premiereleague", "premiereleague-original"),
    ("ctu-premiereleague", "premiereleague-temporal"),
    ("ctu-tpcd", "tpcd-original"),
    ("ctu-webkp", "webkp-original"),
]

### Get overall results


In [23]:
def get_topn_results(df, n=5):
    df = df[
        [
            "dataset_name",
            "task_name",
            "val_roc_auc",
            "test_roc_auc",
            "val_macro_f1",
            "test_macro_f1",
        ]
    ]
    df = df[
        (
            (df.val_roc_auc >= 0)
            & (df.val_roc_auc <= 1)
            & (df.test_roc_auc >= 0)
            & (df.test_roc_auc <= 1)
        )
        | (
            (df.val_macro_f1 >= 0)
            & (df.val_macro_f1 <= 1)
            & (df.test_macro_f1 >= 0)
            & (df.test_macro_f1 <= 1)
        )
    ]
    df_topn = []
    for val_metric in ["val_macro_f1", "val_roc_auc"]:
        # Drop rows where the validation metric is missing
        df_filtered = df.dropna(subset=[val_metric])

        # For each (dataset_name, task_name), get top N rows by validation metric
        df_topn.append(
            df_filtered.sort_values(val_metric, ascending=False)
            .groupby(["dataset_name", "task_name"], group_keys=False)
            .head(n)
        )

    return (
        pd.concat(df_topn)
        .groupby(["dataset_name", "task_name"])
        .where()
        .aggregate(["mean", "std"])
        .reindex(index=dataset_tasks_index)
    )

In [24]:
lightgbm_df = pd.read_csv("./data/pelesjak_lightgbm_hyperparams.csv")
lightgbm_agg_df = pd.read_csv("./data/pelesjak_lightgbm_aggregate_hyperparams.csv")

prop_df = pd.read_csv("./data/pelesjak_getml_xgboost.csv")
prop_df["_start_time"] = pd.to_datetime(prop_df["_start_time"], unit="ms")
prop_df_new = prop_df[(prop_df["_start_time"] > pd.Timestamp("2025-10-08 13:30:00"))]
prop_df_old = prop_df[prop_df["task_name"].str.contains("original")]
prop_df = pd.concat([prop_df_new, prop_df_old])


tabular_df = pd.read_csv("./data/pelesjak_resnet_tabular_hyperparams.csv")
tabular_agg_df = pd.read_csv("./data/pelesjak_resnet_tabular_aggregate_hyperparams.csv")

linear_sage_df = pd.read_csv("./data/pelesjak_linear_sage_hyperparams.csv")
resnet_sage_df = pd.read_csv("./data/pelesjak_resnet_sage_hyperparams.csv")
dbformer_df = pd.read_csv("./data/pelesjak_linear_dbformer_hyperparams.csv")

N = 1
lightgbm_df = get_topn_results(lightgbm_df, n=N)
lightgbm_agg_df = get_topn_results(lightgbm_agg_df, n=N)
prop_df = get_topn_results(prop_df, n=N)
tabular_df = get_topn_results(tabular_df, n=N)
tabular_agg_df = get_topn_results(tabular_agg_df, n=N)
linear_sage_df = get_topn_results(linear_sage_df, n=N)
resnet_sage_df = get_topn_results(resnet_sage_df, n=N)
dbformer_df = get_topn_results(dbformer_df, n=N)

AttributeError: 'DataFrameGroupBy' object has no attribute 'where'

In [ ]:
overall_df_mean = pd.concat(
    [
        lightgbm_df,
        lightgbm_agg_df,
        prop_df,
        tabular_df,
        tabular_agg_df,
        linear_sage_df,
        resnet_sage_df,
        dbformer_df,
    ],
    axis=1,
    keys=(
        "LightGBM",
        "LightGBM Agg",
        "GetML XGBoost",
        "Tabular ResNet",
        "Tabular ResNet Agg",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ),
)
overall_df_mean = overall_df_mean.reset_index().rename(
    {"dataset_name": "Dataset", "task_name": "Task"}, axis=1
)

overall_df_mean.set_index(["Dataset", "Task"], drop=True, inplace=True)

bin_overall = (
    overall_df_mean.swaplevel(0, 1, axis=1)[["val_roc_auc", "test_roc_auc"]]
    .dropna(how="all")
    .swaplevel(0, 2, axis=1)["mean"]
    .sort_index(axis=1, level=[0, 1], ascending=[True, False])
    .swaplevel(0, 1, axis=1)
    .rename(
        columns={
            "val_roc_auc": "val",
            "test_roc_auc": "test",
        }
    )
    .swaplevel(0, 1, axis=1)
)[
    [
        "LightGBM",
        "LightGBM Agg",
        "GetML XGBoost",
        "Tabular ResNet",
        "Tabular ResNet Agg",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ]
]

mlt_overall = (
    overall_df_mean.swaplevel(0, 1, axis=1)[["val_macro_f1", "test_macro_f1"]]
    .dropna(how="all")
    .swaplevel(0, 2, axis=1)["mean"]
    .sort_index(axis=1, level=[0, 1], ascending=[True, False])
    .swaplevel(0, 1, axis=1)
    .rename(
        columns={
            "val_macro_f1": "val",
            "test_macro_f1": "test",
        }
    )
    .swaplevel(0, 1, axis=1)
)[
    [
        "LightGBM",
        "LightGBM Agg",
        "GetML XGBoost",
        "Tabular ResNet",
        "Tabular ResNet Agg",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ]
]

overall_df_mean = pd.concat([bin_overall, mlt_overall], axis=0)

overall_df_mean = overall_df_mean.round(3)

overall_df_mean_all = overall_df_mean
overall_df_mean = overall_df_mean[
    [
        "LightGBM",
        "GetML XGBoost",
        "Tabular ResNet",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ]
]

ranks_total = (
    overall_df_mean.swaplevel(0, 1, axis=1)["test"]
    .dropna(how="all")
    .rank(ascending=False, axis=1, na_option="bottom", method="average")
)


ranks_merged = overall_df_mean.swaplevel(0, 1, axis=1)["test"].dropna(how="all")
ranks_merged["RDL"] = ranks_merged[["Linear SAGE", "ResNet SAGE", "DBFormer"]].max(axis=1)
ranks_merged = ranks_merged[["LightGBM", "GetML XGBoost", "Tabular ResNet", "RDL"]].rank(
    ascending=False, axis=1, na_option="bottom", method="average"
)
ranks_merged_vals = np.zeros(6)
ranks_merged_vals[:4] = ranks_merged.mean(axis=0).values
# overall_df_mean = overall_df_mean.swaplevel(0, 1, axis=1)
# overall_df_mean.loc[("Avg. Rank", ""), "test"] = ranks_total.mean(axis=0).values
# overall_df_mean.loc[("Avg. Rank Merged", ""), "test"] = ranks_merged_vals
# overall_df_mean = overall_df_mean.swaplevel(0, 1, axis=1)


def highlight_max_val_test(row):
    is_val = row.index.get_level_values(1) == "val"
    is_test = row.index.get_level_values(1) == "test"

    styles = pd.Series("", index=row.index)

    val_max = row[is_val].max()
    styles.loc[(row == val_max) & is_val] = "font-weight: bold;"

    test_max = row[is_test].max()
    styles.loc[(row == test_max) & is_test] = "font-weight: bold;"

    return styles


overall_df_mean = overall_df_mean.reset_index()
# ctu_mask = overall_df_mean["Dataset"].str.contains("ctu")
# overall_df_mean.loc[ctu_mask, "Task"] = (
#     overall_df_mean.loc[ctu_mask, "Task"].str.split("-").str[1]
# ).values
# overall_df_mean["Dataset"] = (
#     overall_df_mean["Dataset"].str.replace("ctu-", "").str.replace("rel-", "")
# )

overall_df_mean.set_index(["Dataset", "Task"], drop=True, inplace=True)
overall_df_mean.style.apply(highlight_max_val_test, axis=1).to_latex(
    "./data/tables/overall_results.tex",
    convert_css=True,
    column_format="llcccccccccccc",
)

overall_df_mean

LightGBM        GetML XGBoost  \
                                                val   test           val   
Dataset            Task                                                    
ctu-ergastf1       ergastf1-original         0.5790 0.6050        0.9170   
ctu-expenditures   expenditures-original     0.8520 0.8560        0.8130   
ctu-geneea         geneea-original           0.9940 0.9930        0.9370   
                   geneea-temporal           0.9970 0.9900        0.9450   
ctu-hepatitis      hepatitis-original        0.6660 0.6260        0.9640   
ctu-imdb           imdb-original             0.9860 0.9860        0.5490   
ctu-mondial        mondial-original          0.5000 0.5000           NaN   
ctu-movielens      movielens-original        0.5830 0.6110        0.7840   
ctu-musklarge      musklarge-original        0.5000 0.5000        0.6190   
ctu-musksmall      musksmall-original        0.5000 0.5000        0.8000   
ctu-mutagenesis    mutagenesis-original      0.9170 0.8210        0.9170   
ctu-ncaa           ncaa-original             0.6870 0.4260        0.6540   
ctu-studentloan    studentloan-original      0.5000 0.5000        0.7280   
rel-amazon         item-churn                0.7380 0.7400           NaN   
                   user-churn                0.5790 0.5790           NaN   
rel-avito          user-clicks               0.5650 0.5390           NaN   
                   user-visits               0.5360 0.5280           NaN   
rel-f1             driver-dnf                0.6840 0.6850        0.6800   
                   driver-top3               0.7290 0.7280        0.6810   
rel-stack          user-badge                0.7850 0.7610        0.5140   
                   user-engagement           0.8420 0.8320        0.5890   
rel-trial          study-outcome             0.6760 0.7110        0.6310   
ctu-accidents      accidents-original        0.3790 0.3700        0.4950   
                   accidents-temporal        0.1860 0.1700        0.3210   
ctu-craftbeer      craftbeer-original        0.4110 0.2660        0.0060   
ctu-dallas         dallas-original           0.5940 0.5810        0.5500   
                   dallas-temporal           0.5700 0.5840        0.6670   
ctu-diabetes       diabetes-original         0.1900 0.1900        0.4020   
ctu-financial      financial-original        0.4610 0.4580        0.9350   
                   financial-temporal        0.4950 0.4920        0.6340   
ctu-genes          genes-original            0.0600 0.0600        0.0660   
ctu-hockey         hockey-original           0.7070 0.7210        0.6890   
ctu-legalacts      legalacts-original        0.9330 0.9240        0.7260   
                   legalacts-temporal        0.8700 0.8510        0.7270   
ctu-premiereleague premiereleague-original   0.3800 0.4110        0.6300   
                   premiereleague-temporal   0.3170 0.2220        0.6470   
ctu-tpcd           tpcd-original             0.1910 0.1820        0.1930   
ctu-webkp          webkp-original            0.1390 0.1300        0.3500   

                                                  Tabular ResNet         \
                                             test            val   test   
Dataset            Task                                                   
ctu-ergastf1       ergastf1-original       0.9230         0.5370 0.5970   
ctu-expenditures   expenditures-original   0.8130         0.8450 0.8470   
ctu-geneea         geneea-original         0.9530         0.9900 0.9830   
                   geneea-temporal         0.9340         0.9880 0.9820   
ctu-hepatitis      hepatitis-original      0.9320         0.7000 0.6330   
ctu-imdb           imdb-original           0.5460         0.9840 0.9850   
ctu-mondial        mondial-original           NaN         0.5000 0.5420   
ctu-movielens      movielens-original      0.8060         0.5380 0.6090   
ctu-musklarge      musklarge-original      0.7600         0.5000 0.7000   
ctu-musksmall      musksma

In [25]:
compare_tasks_index = [
    ("ctu-ergastf1", "ergastf1-original"),
    ("ctu-hepatitis", "hepatitis-original"),
    ("ctu-mondial", "mondial-original"),
    ("ctu-movielens", "movielens-original"),
    ("ctu-ncaa", "ncaa-original"),
    ("ctu-studentloan", "studentloan-original"),
    ("rel-amazon", "user-churn"),
    ("rel-avito", "user-clicks"),
    ("rel-avito", "user-visits"),
    ("rel-stack", "user-badge"),
    ("ctu-accidents", "accidents-original"),
    ("ctu-accidents", "accidents-temporal"),
    ("ctu-craftbeer", "craftbeer-original"),
    ("ctu-diabetes", "diabetes-original"),
    ("ctu-genes", "genes-original"),
    ("ctu-premiereleague", "premiereleague-original"),
    ("ctu-premiereleague", "premiereleague-temporal"),
    ("ctu-tpcd", "tpcd-original"),
    ("ctu-webkp", "webkp-original"),
]
tabular_compare_df_mean = overall_df_mean_all[
    [
        "LightGBM",
        "LightGBM Agg",
        "Tabular ResNet",
        "Tabular ResNet Agg",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ]
]
tabular_compare_df_mean.loc[compare_tasks_index]

LightGBM        LightGBM Agg  \
                                                val   test          val   
Dataset            Task                                                   
ctu-ergastf1       ergastf1-original         0.5790 0.6050       0.9190   
ctu-hepatitis      hepatitis-original        0.6660 0.6260       0.9400   
ctu-mondial        mondial-original          0.5000 0.5000       1.0000   
ctu-movielens      movielens-original        0.5830 0.6110       0.5740   
ctu-ncaa           ncaa-original             0.6870 0.4260       0.7970   
ctu-studentloan    studentloan-original      0.5000 0.5000       1.0000   
rel-amazon         user-churn                0.5790 0.5790       0.5770   
rel-avito          user-clicks               0.5650 0.5390       0.5640   
                   user-visits               0.5360 0.5280       0.5360   
rel-stack          user-badge                0.7850 0.7610       0.7830   
ctu-accidents      accidents-original        0.3790 0.3700       0.3770   
                   accidents-temporal        0.1860 0.1700       0.1770   
ctu-craftbeer      craftbeer-original        0.4110 0.2660       0.4880   
ctu-diabetes       diabetes-original         0.1900 0.1900       0.1900   
ctu-genes          genes-original            0.0600 0.0600       0.0600   
ctu-premiereleague premiereleague-original   0.3800 0.4110       0.4700   
                   premiereleague-temporal   0.3170 0.2220       0.5270   
ctu-tpcd           tpcd-original             0.1910 0.1820       0.8460   
ctu-webkp          webkp-original            0.1390 0.1300       0.1390   

                                                  Tabular ResNet         \
                                             test            val   test   
Dataset            Task                                                   
ctu-ergastf1       ergastf1-original       0.9040         0.5370 0.5970   
ctu-hepatitis      hepatitis-original      0.9130         0.7000 0.6330   
ctu-mondial        mondial-original        0.8150         0.5000 0.5420   
ctu-movielens      movielens-original      0.6220         0.5380 0.6090   
ctu-ncaa           ncaa-original           0.7760         0.6150 0.5470   
ctu-studentloan    studentloan-original    1.0000         0.5000 0.5320   
rel-amazon         user-churn              0.5770         0.5260 0.5310   
rel-avito          user-clicks             0.5560         0.5400 0.5410   
                   user-visits             0.5320         0.5180 0.5120   
rel-stack          user-badge              0.7590         0.6890 0.6760   
ctu-accidents      accidents-original      0.3720         0.3540 0.3550   
                   accidents-temporal      0.1630         0.2020 0.1870   
ctu-craftbeer      craftbeer-original      0.2450         0.4120 0.2310   
ctu-diabetes       diabetes-original       0.1900         0.1890 0.1900   
ctu-genes          genes-original          0.0600         0.0600 0.0600   
ctu-premiereleague premiereleague-original 0.4160         0.4890 0.3620   
                   premiereleague-temporal 0.4850         0.4940 0.3700   
ctu-tpcd           tpcd-original           0.8410         0.1910 0.1850   
ctu-webkp          webkp-original          0.1300         0.1390 0.1300   

                                           Tabular ResNet Agg         \
                                                          val   test   
Dataset            Task                                                
ctu-ergastf1       ergastf1-original                   0.8820 0.8930   
ctu-hepatitis      hepatitis-original                  0.6960 0.6320   
ctu-mondial        mondial-original                    0.9870 0.8980   
ctu-movielens      movielens-original                  0.5380 0.6090   
ctu-ncaa           ncaa-original                       0.6760 0.7410   
ctu-studentloan    studentloan-original                1.0000 1.0000   
rel-amazon         user-churn                          0.5260 0.5310   
rel-avito          us

In [26]:
dataset_info = pd.read_csv("../redelex/datasets/dataset-info.csv")
task_info = pd.read_csv("../redelex/tasks/task-info.csv")

_joined_info = task_info.merge(dataset_info, how="left", on=["dataset"]).set_index(
    ["dataset", "task"], drop=True
)
joined_info = _joined_info.select_dtypes(include="number")
joined_info = joined_info.loc[overall_df_mean.index]

ordered_features = {
    "n_tables": "#Tab.",
    "n_fks": "#FK",
    "n_factual_cols": "#Factual",
    "cat_col": "#Cat.",
    "num_col": "#Num.",
    "text_col": "#Text",
    "time_col": "#Time",
    "total_n_tuples": "#Rows",
    "total_n_fk_edges": "#Links",
    "schema_diameter": "Diameter",
    "one_to_one": "1-to-1",
    "one_to_many": "1-to-M",
    "n_train": "#Train",
    "entity_fact_cols": "#T. Factual",
    "target_categorical": "T. Cat.",
    "target_numerical": "T. Num.",
    "target_text_embedded": "T. Text",
    "target_timestamp": "T. Time",
    "entity_eccentricity": "Eccentricity",
    "sample_density": "Density",
}
joined_info = joined_info[ordered_features.keys()].rename(columns=ordered_features)
all_joined_info = _joined_info[ordered_features.keys()].rename(columns=ordered_features)

In [27]:
best_ranks = ranks_total.min(axis=1)
rdl_best = ranks_total[
    (ranks_total["Linear SAGE"] == best_ranks)
    | (ranks_total["ResNet SAGE"] == best_ranks)
    | (ranks_total["DBFormer"] == best_ranks)
]

tabular_best = ranks_total[
    (ranks_total["Tabular ResNet"] == best_ranks) | (ranks_total["LightGBM"] == best_ranks)
]

prop_best = ranks_total[(ranks_total["GetML XGBoost"] == best_ranks)]

rdl_info = joined_info.loc[rdl_best.index]
tabular_info = joined_info.loc[tabular_best.index]
prop_info = joined_info.loc[prop_best.index]

col_map = {0.25: "Q1", 0.5: "Median", 0.75: "Q3"}
base_info = all_joined_info.quantile([0.5]).T.rename(columns=col_map)
prop_info = prop_info.quantile([0.25, 0.5, 0.75]).T.rename(columns=col_map)
rdl_info = rdl_info.quantile([0.25, 0.5, 0.75]).T.rename(columns=col_map)
tabular_info = tabular_info.quantile([0.25, 0.5, 0.75]).T.rename(columns=col_map)

features_summary = pd.concat(
    [base_info, rdl_info, prop_info, tabular_info],
    axis=1,
    keys=["Base", "RDL", "Propositional.", "Tabular Learning"],
)

features_summary.to_latex(
    "./data/tables/features_summary.tex",
    column_format="lcccccccccc",
    float_format="%.3f",
)
features_summary

Base        RDL                           Propositional.  \
                  Median         Q1       Median           Q3             Q1   
#Tab.             6.0000     3.0000       7.0000       8.5000         3.5000   
#FK               6.0000     2.0000       6.0000      12.0000         4.0000   
#Factual         35.0000    16.5000      33.0000      83.5000        23.0000   
#Cat.            11.0000     2.5000      11.0000      21.5000        11.5000   
#Num.             6.0000     1.0000       6.0000      11.5000         3.5000   
#Text             6.0000     1.0000       6.0000      13.0000         0.5000   
#Time             2.0000     0.0000       1.0000       4.0000         0.5000   
#Rows        472489.0000 17212.0000 1052086.0000 5523756.0000     11308.0000   
#Links       752362.0000 28023.0000 1032369.0000 7805732.5000     31867.0000   
Diameter          3.0000     2.0000       2.0000       4.0000         2.0000   
1-to-1            0.0000     0.0000       0.0000       4.0000         0.0000   
1-to-M            5.0000     2.0000       3.0000      11.0000         4.0000   
#Train        11994.0000   745.0000   15774.0000  530684.0000       259.0000   
#T. Factual       5.0000     1.0000       4.0000       7.0000         2.5000   
T. Cat.           2.0000     0.0000       0.0000       2.5000         1.0000   
T. Num.           0.0000     0.0000       0.0000       1.0000         0.0000   
T. Text           1.0000     0.0000       1.0000       3.5000         0.0000   
T. Time           1.0000     0.0000       0.0000       1.0000         0.0000   
Eccentricity     10.6400     5.9600      12.2400      18.0900         9.5300   
Density           0.0247     0.0057       0.0216       0.1280         0.0010   

                                      Tabular Learning               \
                  Median           Q3               Q1       Median   
#Tab.             7.0000       8.5000           4.0000       4.0000   
#FK               6.0000      10.5000           4.0000       4.0000   
#Factual         47.0000      93.5000          28.0000      28.0000   
#Cat.            17.0000      66.0000          13.0000      13.0000   
#Num.             5.0000      12.5000           3.0000       3.0000   
#Text             2.0000       6.0000           3.0000      14.0000   
#Time             1.0000       6.0000           3.0000       5.0000   
#Rows         97606.0000  812773.0000      803901.0000 1754299.0000   
#Links       227716.0000 1084972.0000      850783.0000 2340922.0000   
Diameter          3.0000       3.0000           3.0000       3.0000   
1-to-1            0.0000       0.0000           0.0000       0.0000   
1-to-M            5.0000       9.5000           4.0000       4.0000   
#Train          524.0000    1027.5000        2818.0000   11994.0000   
#T. Factual       5.0000       5.0000          15.0000      20.0000   
T. Cat.           2.0000       3.0000          10.0000      10.0000   
T. Num.           0.0000       2.0000           2.0000       3.0000   
T. Text           0.0000       0.0000           2.0000       2.0000   
T. Time           1.0000       1.0000           1.0000       2.0000   
Eccentricity     13.6800      18.4400           5.7800      10.5800   
Density           0.0047       0.0301           0.0093       0.0783   

                           
                       Q3  
#Tab.             15.0000  
#FK               15.0000  
#Factual         110.0000  
#Cat.             44.0000  
#Num.              6.0000  
#Text             32.0000  
#Time              5.0000  
#Rows        1754299.0000  
#Links       2340922.0000  
Diameter           4.0000  
1-to-1             4.0000  
1-to-M            13.0000  
#Train        452352.0000  
#T. Factual       20.0000  
T. Cat.           12.0000  
T. Num.            3.0000  
T. Text            6.0000  
T. Time            5.0000  
Eccentricity      10.6400  
Density            0.1388